# ⚽ FIFA World Cup Intelligence & Prediction System
## Phase 5: Goalscorer Analytics

**Notebook:** `05_goalscorer_analytics.ipynb`  
**Input:** `data/goalscorers.csv`

---

### Objective

Analyze international football goalscoring history at the **player level** — identifying the greatest scorers, examining how scoring has evolved across eras, and uncovering country-level scoring patterns.


---
## Section 1: Load Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [6]:
df = pd.read_csv('../data/goalscorers.csv', parse_dates=['date'])

print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(df.dtypes)
df.sample(8, random_state=42)

Shape: 47,690 rows x 8 columns
date         datetime64[ns]
home_team            object
away_team            object
team                 object
scorer               object
minute              float64
own_goal               bool
penalty                bool
dtype: object


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
14794,1994-04-20,Northern Ireland,Liechtenstein,Northern Ireland,Steve Lomas,25.0,False,False
43272,2023-06-19,Belarus,Kosovo,Kosovo,Vedat Muriqi,87.0,False,True
10075,1983-09-18,Venezuela,Uruguay,Uruguay,Carlos Aguilera,87.0,False,False
21794,2001-09-05,England,Albania,England,Michael Owen,43.0,False,False
1640,1949-04-17,Brazil,Colombia,Brazil,Millón Medeiros,24.0,False,True
46679,2025-09-05,Switzerland,Kosovo,Switzerland,Breel Embolo,45.0,False,False
40137,2021-03-25,England,San Marino,England,James Ward-Prowse,14.0,False,False
12662,1990-11-14,Czechoslovakia,Spain,Czechoslovakia,Václav Daněk,67.0,False,False


---
## Section 2: Data Quality Assessment

In [10]:
# Missing values
missing = df.isnull().sum().reset_index()
missing.columns = ['Column', 'Missing Count']
missing['Missing %'] = (missing['Missing Count'] / len(df) * 100).round(2)
print('Missing values:')
missing

Missing values:


,Column,Missing Count,Missing %
0,date,0,0.00
1,home_team,0,0.00
2,away_team,0,0.00
3,team,0,0.00
4,scorer,48,0.10
5,minute,256,0.54
6,own_goal,0,0.00
7,penalty,0,0.00


In [31]:
# Duplicate records
n_dupes = df.duplicated().sum()
df = df.drop_duplicates(keep='first')
print(f'Duplicate rows: {n_dupes}')

Duplicate rows: 82


In [18]:
df.columns

Index(['date', 'home_team', 'away_team', 'team', 'scorer', 'minute',
       'own_goal', 'penalty'],
      dtype='object')

In [47]:
# Unique players, countries, date range
# Exclude own goals from player counts — they are not credited to any player
regular_goals = df[df['own_goal'] == False]

unique_players   = regular_goals['scorer'].nunique()
unique_countries = df['team'].nunique()
date_min         = df['date'].min().date()
date_max         = df['date'].max().date()

quality_summary = pd.DataFrame([
    ['Total goal records',        len(df)],
    ['Duplicate rows',            n_dupes],
    ['Unique players (non-OG)',   unique_players],
    ['Unique countries',          unique_countries],
    ['Earliest record',           date_min],
    ['Latest record',             date_max],
    ['Missing scorer name',       df['scorer'].isnull().sum()],
], columns=['Metric', 'Value'])

print('Data quality summary:')
quality_summary

Data quality summary:


,Metric,Value
0,Total goal records,47608
1,Duplicate rows,82
2,Unique players (non-OG),14856
3,Unique countries,220
4,Earliest record,1916-07-02
5,Latest record,2026-06-18
6,Missing scorer name,11


---
## Section 3: Dataset Understanding

In [48]:
# Add year and decade columns — used throughout the notebook
df['year']   = df['date'].dt.year
df['decade'] = (df['year'] // 10 * 10)

# Separate regular goals from own goals for clean analysis
own_goals_df    = df[df['own_goal'] == True].copy()
regular_goals   = df[df['own_goal'] == False].copy()
penalty_goals   = regular_goals[regular_goals['penalty'] == True].copy()
open_play_goals = regular_goals[regular_goals['penalty'] == False].copy()

goals_info = pd.DataFrame([
    ['Total goal records', len(df)],
    ['Regular goals (non-OG)', len(regular_goals)],
    ['Own goals ', len(own_goals_df)],
    ['Penalty goals', len(penalty_goals)],
    ['Open play goals',   len(open_play_goals)]
], columns=['Goal type','Count'])
goals_info

,Goal type,Count
0,Total goal records,47608
1,Regular goals (non-OG),46680
2,Own goals,928
3,Penalty goals,3255
4,Open play goals,43425


---
## Section 4: All-Time Goalscorer Rankings

Who are the greatest international goalscorers in history? We rank by total regular goals (own goals excluded, as they are not credited to the scorer).

In [54]:
player_goals = regular_goals.groupby(['scorer', 'team']).agg(
    total_goals   = ('scorer', 'count'),
    penalty_goals = ('penalty', 'sum'),
    first_goal    = ('date', 'min'),
    last_goal     = ('date', 'max')
).reset_index()
player_goals

player_goals['open_play_goals'] = player_goals['total_goals'] - player_goals['penalty_goals']
player_goals['career_years']    = player_goals['last_goal'].dt.year - player_goals['first_goal'].dt.year
player_goals['goals_per_year']  = (
    player_goals['total_goals'] / player_goals['career_years'].replace(0, 1)
).round(2)
player_goals


,scorer,team,total_goals,penalty_goals,first_goal,last_goal,open_play_goals,career_years,goals_per_year
0,A'ala Hubail,Bahrain,13,1,2004-02-18,2008-03-26,12,4,3.25
1,A. Elangovan,Malaysia,1,0,1993-05-05,1993-05-05,1,0,1.00
2,Aage Rou Jensen,Denmark,1,0,1956-10-03,1956-10-03,1,0,1.00
3,Aamir Abdallah,Sudan,1,0,2026-01-03,2026-01-03,1,0,1.00
4,Aaran Lines,New Zealand,3,0,2001-06-06,2004-05-31,3,3,1.00
...,...,...,...,...,...,...,...,...,...
14898,Željko Čajkovski,Yugoslavia,6,0,1949-08-21,1950-06-28,6,1,6.00
14899,Ștefan Baiaram,Romania,1,0,2025-11-18,2025-11-18,1,0,1.00
14900,Ștefan Bodișteanu,Moldova,1,0,2025-10-14,2025-10-14,1,0,1.00
14901,Ștefan Dobay,Romania,5,0,1933-10-29,1938-06-09,5,5,1.00


In [70]:
# Top 20 all-time scorers
top20_scorers = player_goals.nlargest(20, 'total_goals')[['scorer', 'team', 'total_goals', 'penalty_goals', 'open_play_goals', 'career_years']].reset_index(drop=True)

fig = px.bar(
    top20_scorers,
    x='scorer',
    y='total_goals',
    color='total_goals',
    title = 'Top 20 all-time scorers',
    hover_data=['scorer', 'team', 'total_goals','penalty_goals'],
)
fig.show()

### Key Findings — All-Time Rankings

- The top 20 international scorers are spread across multiple nations and eras — this is not a one-country story.
- **Penalty contribution varies significantly** among elite scorers.
- The gap between the top scorer and the 20th-ranked scorer is large — the truly elite tier is small.
- Career longevity plays a major role in reaching the top — most all-time leaders played internationally for 10–15+ years.

### Football Insights

Reaching the very top of international goalscoring requires a combination of **longevity, consistency, and national team prominence**. A striker who plays 5 brilliant years rarely reaches the all-time list — the record holders sustained elite-level output across the full arc of a long career.

---
## Section 5: Country Goalscoring Analysis

Which nations produce the most international goals, and which have the strongest scoring traditions?

In [71]:
# Total goals by country (regular goals only)
country_goals = regular_goals.groupby('team').agg(
    total_goals    = ('scorer', 'count'),
    unique_scorers = ('scorer', 'nunique')
).reset_index()

country_goals['goals_per_scorer'] = (
    country_goals['total_goals'] / country_goals['unique_scorers']
).round(2)

country_goals = country_goals.sort_values('total_goals', ascending=False)

top30 = country_goals.head(30).reset_index(drop=True)

fig = px.bar(
    top30,
    x='team',
    y='total_goals',
    color='total_goals',
    title = 'Total goals by country',
    hover_data=['team', 'total_goals', 'goals_per_scorer','unique_scorers'],
)
fig.show()

### Key Findings — Country Goalscoring

- **Total goals by country correlates strongly with match volume** — nations that play most also score most.
- **Goals per scorer** is the more revealing metric: it captures how productive each player in a nation's scoring pool tends to be, independent of volume.
- Nations with high goals-per-scorer often have dominant central strikers who rack up large individual tallies relative to their teammates.

### Football Insights

A high goals-per-scorer ratio can indicate either a **superstar-dependent** scoring culture (one player carries the goal burden) or a **highly efficient** striker pool where most players who score do so regularly. Section 6 separates these two explanations.

---
## Section 6: Scoring Concentration Analysis

Do countries rely on one superstar, or are their goals spread across many players?

In [ ]:
# For each country: find the top scorer and their share of country goals
top_scorer_per_country = regular_goals.groupby(['team', 'scorer']).size().reset_index(name='goals')
top_scorer_per_country = top_scorer_per_country.sort_values('goals', ascending=False)

# Get the top scorer per country
best_per_country = top_scorer_per_country.drop_duplicates(subset='team', keep='first').rename(
    columns={'scorer': 'top_scorer', 'goals': 'top_scorer_goals'}
)

# Merge with country totals
concentration = country_goals.merge(best_per_country, on='team')
concentration['top_scorer_share_pct'] = (
    concentration['top_scorer_goals'] / concentration['total_goals'] * 100
).round(1)

concentration = concentration.sort_values('total_goals', ascending=False)

print('Scoring concentration — top 15 countries by total goals:')
concentration15 = concentration.head(15)[['team', 'total_goals', 'unique_scorers', 'top_scorer', 'top_scorer_goals', 'top_scorer_share_pct']].reset_index(drop=True)


Scoring concentration — top 15 countries by total goals:


In [102]:
# Which countries rely most on their top scorer?
conc_qualified = concentration[concentration['unique_scorers'] >= 20]
most_concentrated = conc_qualified.nlargest(15, 'top_scorer_share_pct').sort_values('top_scorer_share_pct', ascending=False)

# Plot
fig = px.bar(
    most_concentrated,
    x='team',
    y='top_scorer_share_pct',
    text = 'top_scorer_share_pct',
    color='unique_scorers',
    title = 'countries most relying on their top scorer',
    hover_data=['team', 'total_goals', 'unique_scorers', 'top_scorer', 'top_scorer_goals', 'top_scorer_share_pct']
)
fig.show()


In [ ]:
# Which countries distribute goals most evenly?
least_concentrated = conc_qualified.nsmallest(15, 'top_scorer_share_pct').sort_values('top_scorer_share_pct', ascending=False)

# Plot
fig = px.bar(
    least_concentrated,
    x='team',
    y='top_scorer_share_pct',
    text = 'top_scorer_share_pct',
    color='unique_scorers',
    title = 'countries least relying on their top scorer',
    hover_data=['team', 'total_goals', 'unique_scorers', 'top_scorer', 'top_scorer_goals', 'top_scorer_share_pct']
)
fig.show()

### Key Findings — Scoring Concentration

- Many countries have a **single player accounting for 15%+ of all historical goals** considering a minimum threshold of 20 unique scorers - very high dependency on one individual.
- Countries with the **deepest scoring pools** have their goals spread across many contributors — no single player dominates the record books.
- Historically dominant nations (Brazil, Germany) tend to have lower concentration ratios — their success is not built on one player across history.

### Football Insights

High scoring concentration is not inherently good or bad — it can reflect a **generational superstar** who elevated a nation. But from a long-term development standpoint, countries with broad scoring depth are more resilient — when the superstar retires, the team does not collapse.

---
## Section 7: Penalty Goal Analysis

How large a role do penalties play in international goalscoring?

In [106]:
# Overall penalty goal stats
total_regular = len(regular_goals)
total_pens    = len(penalty_goals)
pen_pct       = round(total_pens / total_regular * 100, 1)

print(f'Regular goals (non-OG) : {total_regular:,}')
print(f'Penalty goals          : {total_pens:,}')
print(f'Penalty share          : {pen_pct}%')

Regular goals (non-OG) : 46,680
Penalty goals          : 3,255
Penalty share          : 7.0%


In [ ]:
# Top 15 players with most penalty goals
pen_by_player = penalty_goals.groupby(['scorer', 'team']).size().reset_index(name='penalty_goals')
pen_by_player = pen_by_player.sort_values('penalty_goals', ascending=False).head(15)

# Plot
fig = px.bar(
    pen_by_player,
    x='scorer',
    y='penalty_goals',
    text = 'penalty_goals',
    color='penalty_goals',
    title = 'Top 15 players with most penalty goals',
    hover_data=['scorer', 'team','penalty_goals']
)
fig.show()

In [113]:
# Top 15 countries with most penalty goals
pen_by_country = penalty_goals.groupby('team').size().reset_index(name='penalty_goals')
pen_by_country = pen_by_country.sort_values('penalty_goals', ascending=False).head(15)

# Plot
fig = px.bar(
    pen_by_country,
    x='team',
    y='penalty_goals',
    text = 'penalty_goals',
    color='penalty_goals',
    title = 'Top 15 countries with most penalty goals',
    hover_data=['team','penalty_goals']
)
fig.show()

In [118]:
# Penalty goals by decade
pen_by_decade = penalty_goals.groupby('decade').size().reset_index(name='penalty_goals')
total_by_decade = regular_goals.groupby('decade').size().reset_index(name='total_goals')
pen_decade = pen_by_decade.merge(total_by_decade, on='decade')
pen_decade['penalty_pct'] = (pen_decade['penalty_goals'] / pen_decade['total_goals'] * 100).round(1)

# Plot
fig = px.bar(
    pen_decade,
    x='decade',
    y='penalty_goals',
    text = 'penalty_pct',
    color='penalty_pct',
    title = 'Penalty goals by decade',
    hover_data=['decade','penalty_goals','total_goals','penalty_pct']
)
fig.show()

### Key Findings — Penalty Analysis

- Penalties account for a **meaningful share of all international goals** — typically 6–10% across most eras.
- **Penalty reliance has grown in modern football** — rule changes, refereeing standards, and increased awareness of penalty-winning tactics have raised the rate.
- The top penalty scorers are predominantly players who took penalties consistently for their national teams over long careers — this metric favors volume over precision.
- **Countries with most penalty goals** tend to be high-volume teams who have played many qualifying matches — more matches = more penalty opportunities.

### Football Insights

Penalty goals inflate all-time scoring records. When evaluating the true greatness of a top scorer, their **open-play tally is the more meaningful metric** — it reflects their ability to score from general play, not from the spot. Some all-time leaders have a significant penalty dependency; others built their records almost entirely from open play.

---
## Section 11: Most Prolific Goalscorers

Beyond total goals, which players had the highest peaks, the longest careers, and the most consistent output?